# Autonomous RAG Notebook : LangGraph + Groq + Hugging Face

## Concept: Retrieval-Augmented Generation (RAG) with Chain-of-Thought (CoT)

This notebook demonstrates how to build an **Autonomous RAG pipeline** using the **LangGraph framework** (2026), **Groq LLMs**, and **Hugging Face embeddings**.  
The goal is to answer complex questions about a knowledge source (here, the *Xumo Stream Box manual*) by combining **retrieval** and **reasoning**.

- **RAG**: Retrieval-Augmented Generation ensures answers are grounded in actual documentation rather than hallucinations.  
- **CoT**: Chain-of-Thought reasoning breaks down queries into smaller steps before synthesizing the final answer.  
- **LangGraph**: Provides a graph-based orchestration where nodes represent planner, retriever, and responder stages.  
- **Groq LLM**: Executes reasoning and synthesis with high performance.  
- **Hugging Face embeddings**: Used to vectorize text chunks for semantic retrieval.  
- **Gradio**: Wraps the pipeline into an interactive interface for querying.

---

## Workflow

1. **Load Manual**: The Xumo Stream Box manual is ingested as text.  
2. **Split Documents**: The text is chunked into overlapping segments for better retrieval.  
3. **Embed & Index**: Hugging Face embeddings are generated and stored in FAISS.  
4. **Plan Steps**: The LLM breaks the user query into reasoning steps.  
5. **Retrieve Context**: Relevant chunks are retrieved per step.  
6. **Generate Answer**: The LLM synthesizes a final answer using retrieved context.  
7. **Gradio UI**: Provides a textbox for queries and displays reasoning steps + final answer.

---

## Example Query

**User Question:**  
*Is the Xumo Stream Box 4K?*

---

### Reasoning Steps
1. **Check Specifications**  
   Inspect the manual for resolution support.  
2. **Look for 4K Mention**  
   Verify if 3840×2160 resolution or 4K HDR formats are supported.  
3. **Check Connected Device**  
   Ensure the TV/monitor also supports 4K to realize the capability.

---

### Final Answer
Based on the **September 2025 Xumo Stream Box User Manual**:

- The device supports **Dolby Vision HDR** and **Dolby Atmos audio**.  
- Dolby Vision is a widely used **4K HDR format**, strongly suggesting 4K capability.  
- HDMI connectivity is mentioned, which is the standard interface for 4K resolution.  
- Picture settings allow resolution configuration, implying support for higher resolutions.

**Conclusion:**  
The Xumo Stream Box is capable of supporting **4K resolution**, most likely through Dolby Vision and HDMI features.

---

## Why This Matters
This notebook shows how **Autonomous RAG agents** can:
- Ground answers in official documentation.  
- Provide **transparent reasoning steps**.  
- Deliver **interactive exploration** via Gradio.  

It’s a practical teaching example of how modern GenAI pipelines combine **retrieval, reasoning, and user interfaces** to answer technical questions reliably.

In [ ]:
# -----------------------------
# 0. Setup & Imports
# -----------------------------
import os
from typing import List
from pydantic import BaseModel
from dotenv import load_dotenv
import gradio as gr

from langchain_core.documents import Document   # updated import
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings   # Hugging Face embeddings
from langgraph.graph import StateGraph, END


In [5]:
# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [6]:
# -----------------------------
# 1. Prepare Vectorstore
# -----------------------------
docs = TextLoader(
    "C:/Users/admin/Desktop/New_GenAI/GenAI/LangGraph/Autonomus RAG/xumo_manual_rag.txt",
    encoding="utf-8"
).load()

In [7]:
# Create a text splitter that breaks the manual into chunks of 500 characters
# with an overlap of 50 characters to preserve context between chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# Apply the splitter to the loaded documents so we get a list of smaller chunks
chunks = splitter.split_documents(docs)

In [8]:
# Hugging Face embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

In [9]:
# -----------------------------
# 2. Initialize Groq LLM
# -----------------------------
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

In [11]:
# -----------------------------
# 3. LangGraph State Definition
# -----------------------------
class ManualRAGState(BaseModel):   
    question: str
    sub_steps: List[str] = []
    retrieved_docs: List[Document] = []
    answer: str = ""

In [17]:
# -----------------------------
# 4. Nodes
# -----------------------------
def plan_steps(state: ManualRAGState) -> ManualRAGState:
    prompt = f"Break the question into 2-3 reasoning steps:\n\n{state.question}"
    result = llm.invoke(prompt).content
    subqs = [line.strip("- ") for line in result.split("\n") if line.strip()]
    return state.model_copy(update={"sub_steps": subqs})

def retrieve_per_step(state: ManualRAGState) -> ManualRAGState:
    all_docs = []
    for sub in state.sub_steps:
        docs = retriever.invoke(sub)
        all_docs.extend(docs)
    return state.model_copy(update={"retrieved_docs": all_docs})

def generate_answer(state: ManualRAGState) -> ManualRAGState:
    context = "\n\n".join([doc.page_content for doc in state.retrieved_docs])
    prompt = f"""
You are answering a complex question using reasoning and retrieved documents.

Question: {state.question}

Relevant Information:
{context}

Now synthesize a well-reasoned final answer.
"""
    result = llm.invoke(prompt).content.strip()
    return state.model_copy(update={"answer": result})


In [13]:
# -----------------------------
# 5. LangGraph Graph
# -----------------------------
builder = StateGraph(ManualRAGState)
builder.add_node("planner", plan_steps)
builder.add_node("retriever", retrieve_per_step)
builder.add_node("responder", generate_answer)

builder.set_entry_point("planner")
builder.add_edge("planner", "retriever")
builder.add_edge("retriever", "responder")
builder.add_edge("responder", END)

graph = builder.compile()


In [15]:
# -----------------------------
# 6. Gradio Interface
# -----------------------------
def rag_pipeline(user_query: str):
    state = ManualRAGState(question=user_query)
    final = graph.invoke(state)
    steps = "\n".join(final["sub_steps"])
    return f"Reasoning Steps:\n{steps}\n\nFinal Answer:\n{final['answer']}"

demo = gr.Interface(
    fn=rag_pipeline,
    inputs=gr.Textbox(label="Ask about Xumo Stream Box Manual"),
    outputs=gr.Textbox(label="RAG Answer"),
    title="Xumo Manual RAG Assistant"
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
